In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
import networkx as nx
from collections import defaultdict
import pandas as pd
from sentence_transformers import SentenceTransformer

from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_distances

In [ ]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
df = pd.read_pickle('df_with_normalized_units.pkl')
#df = df.reset_index(drop=True)
#df['story_id'] = df.index
df.drop(columns=['canonical_units','story_vec'], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1503 entries, 0 to 1502
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        1503 non-null   int64 
 1   atu_id            1503 non-null   object
 2   tale_title        1503 non-null   object
 3   provenance        1489 non-null   object
 4   notes             821 non-null    object
 5   source            1325 non-null   object
 6   text              1503 non-null   object
 7   data_source       1503 non-null   object
 8   date_obtained     1503 non-null   object
 9   atu_number        1503 non-null   int64 
 10  extracted_units   1503 non-null   object
 11  story_id          1503 non-null   int64 
 12  normalized_units  1503 non-null   object
dtypes: int64(3), object(10)
memory usage: 152.8+ KB


In [ ]:
all_units = []
all_story_ids = []

for _, row in df.iterrows():
    units = row["normalized_units"]
    sid = row["story_id"]
    for u in units:
        all_units.append(u)
        all_story_ids.append(int(sid))

len(all_units)

24497

In [ ]:
model = model.to("cuda")

embeddings = model.encode(
    all_units,
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=256,
    device="cuda"
)


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

In [ ]:
import gc
gc.collect()

0

In [ ]:
from sklearn.cluster import AgglomerativeClustering

#pca = PCA(n_components=200)
#reduced_embeddings = pca.fit_transform(embeddings)

svd = TruncatedSVD(n_components=200)
reduced_embeddings = svd.fit_transform(embeddings)

In [ ]:

clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.6,
    metric="cosine",
    linkage="complete",
#    n_clusters=2000,
#    distance_threshold=None,
#    compute_full_tree=False

)

labels3 = clustering.fit_predict(reduced_embeddings)

sil_score = silhouette_score(reduced_embeddings, labels3)

print(sil_score)

0.074633405


In [ ]:
from collections import defaultdict

clusters = defaultdict(list)

for idx, label in enumerate(labels3):
    if label != -1:
        clusters[label].append(idx)

cluster_sizes = {
    cluster_id: len(indices)
    for cluster_id, indices in clusters.items()
}


sorted_clusters = sorted(
    cluster_sizes.items(),
    key=lambda x: x[1],
    reverse=True
)

print(sorted_clusters[:10])



[(np.int64(101), 58), (np.int64(406), 53), (np.int64(571), 44), (np.int64(1314), 42), (np.int64(510), 40), (np.int64(783), 40), (np.int64(122), 39), (np.int64(160), 34), (np.int64(2580), 32), (np.int64(1431), 31)]


In [ ]:
import numpy as np
from collections import defaultdict

labels = np.array(labels3)
story_ids = np.array(all_story_ids)

motif_clusters = []

for cluster_id, indices in clusters.items():
    unique_stories = set(story_ids[indices])
    if len(unique_stories) >= 2:
        motif_clusters.append(cluster_id)

print("Number of motif clusters:", len(motif_clusters))

all_stories = set(story_ids)

stories_with_motif = set()

for cluster_id in motif_clusters:
    indices = clusters[cluster_id]
    stories_with_motif.update(story_ids[indices])

stories_without_motif = all_stories - stories_with_motif

print("Number of stories without motif:", len(stories_without_motif))
print("Percentage:",
      len(stories_without_motif) / len(all_stories))


story_motif_count = defaultdict(set)

for cluster_id in motif_clusters:
    for idx in clusters[cluster_id]:
        story = story_ids[idx]
        story_motif_count[story].add(cluster_id)

motifs_per_story = [len(v) for v in story_motif_count.values()]

print("Average motifs per story:", np.mean(motifs_per_story))



Number of motif clusters: 3752
Number of stories without motif: 4
Percentage: 0.0026613439787092482
Average motifs per story: 9.313542361574383


In [ ]:
def compute_cluster_entropy(embeddings, cluster_labels):

    global_mean = embeddings.mean(axis=0)
    global_var = np.mean(np.sum((embeddings - global_mean) ** 2, axis=1))

    entropies = []

    for cid in np.unique(cluster_labels):
        idx = np.where(cluster_labels == cid)[0]
        cluster_embs = embeddings[idx]

        if len(cluster_embs) <= 1:
            continue

        cluster_mean = cluster_embs.mean(axis=0)
        cluster_var = np.mean(
            np.sum((cluster_embs - cluster_mean) ** 2, axis=1)
        )

        entropies.append(cluster_var / global_var)

    return float(np.mean(entropies)) if entropies else 0.0

def between_cluster_distances(embeddings, cluster_labels, type= 'mean'):
    labels = np.asarray(cluster_labels)
    unique_labels = np.unique(labels)

    centroids = []
    for cid in unique_labels:
        centroids.append(embeddings[labels == cid].mean(axis=0))

    centroids = np.vstack(centroids)

    dist_matrix = cosine_distances(centroids)
    iu = np.triu_indices_from(dist_matrix, k=1)
    distances = dist_matrix[iu]
    if type == 'mean':
        return distances.mean()
    elif type == 'max':
        return distances.max()
    elif type == 'min':
        return distances.min()
    else:
       return distances.mean()


In [ ]:
N = len(all_units)
cluster_labels = labels3
K = len(np.unique(cluster_labels))

compression_ratio = N / K
cluster_entropy = compute_cluster_entropy(
            reduced_embeddings, cluster_labels
        )

between_cluster = between_cluster_distances(
            reduced_embeddings, cluster_labels, type = 'mean'
        )

print(f"compression_ration: {compression_ratio}")
print(f"cluster_entropy: {cluster_entropy}")
print(f"between_cluster: {between_cluster}")

compression_ration: 4.890596925534039
cluster_entropy: 0.34358564019203186
between_cluster: 0.8011595606803894


In [ ]:
def cluster_story_distribution(unit_to_clust):
    rows = []

    for cid, g in unit_to_clust.groupby("cluster_label"):
        story_counts = g["story_id"].value_counts()
        n = story_counts.sum() # n is total number of nuites in this cluster
        m = story_counts.shape[0] # m is the number of unique story IDs of units in this cluster


        p = story_counts.values / n

        entropy = -np.sum(p * np.log(p))
        norm_entropy = entropy / np.log(m) if m > 1 else 0.0

        rows.append({
            "cluster_label": cid,
            "cluster_size": n,
            "story_support": m,
            "story_entropy": entropy,
            "story_entropy_norm": norm_entropy,
            "coverage_ratio": m / n
        })

    return pd.DataFrame(rows)

In [ ]:

from collections import defaultdict

cluster_to_indices = defaultdict(list)
for i, cid in enumerate(cluster_labels):
    cluster_to_indices[cid].append(i)

def get_cluster_representative(indices):
    embs = reduced_embeddings[indices]
    centroid = embs.mean(axis=0, keepdims=True)
    sims = cosine_similarity(embs, centroid).ravel()
    return all_units[indices[np.argmax(sims)]]

unit_to_clust = pd.DataFrame({
    "unit": all_units,
    "cluster_label": cluster_labels,
    "story_id": all_story_ids
})


clusters_info = []
for cid, indices in cluster_to_indices.items():
    embs = reduced_embeddings[indices]

    clusters_info.append({
        "cluster_label": cid,
        "cluster_mean_embedding": embs.mean(axis=0),
        "cluster_representative": get_cluster_representative(indices),
        "cluster_size": len(indices)
    })

clusters_info = pd.DataFrame(clusters_info)
clusters_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5009 entries, 0 to 5008
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   cluster_label           5009 non-null   int64 
 1   cluster_mean_embedding  5009 non-null   object
 2   cluster_representative  5009 non-null   object
 3   cluster_size            5009 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 156.7+ KB


In [ ]:
motif_clusters = clusters_info.merge(
    cluster_story_distribution(unit_to_clust),
    on="cluster_label"
)

motif_clusters = motif_clusters[
    motif_clusters["story_support"] >= 2
]
motif_clusters.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3752 entries, 0 to 4998
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   cluster_label           3752 non-null   int64  
 1   cluster_mean_embedding  3752 non-null   object 
 2   cluster_representative  3752 non-null   object 
 3   cluster_size_x          3752 non-null   int64  
 4   cluster_size_y          3752 non-null   int64  
 5   story_support           3752 non-null   int64  
 6   story_entropy           3752 non-null   float64
 7   story_entropy_norm      3752 non-null   float64
 8   coverage_ratio          3752 non-null   float64
dtypes: float64(3), int64(4), object(2)
memory usage: 293.1+ KB


In [ ]:
units_to_embed = dict(zip(all_units, reduced_embeddings))

In [ ]:
unit_to_cluster = dict(zip(unit_to_clust["unit"], unit_to_clust["cluster_label"]))
motif_cluster_to_centroid = dict(
    zip(motif_clusters["cluster_label"], motif_clusters["cluster_mean_embedding"])
)

def units_to_centroids_with_fallback(units):
    centroids = []
    seen_clusters = set()
    for u in units:
        cid = unit_to_cluster.get(u, None)
        if cid is None or cid in seen_clusters:
            continue
        seen_clusters.add(cid)
        centroid = motif_cluster_to_centroid.get(cid, None)
        if centroid is not None:
            centroids.append(centroid)
    if not centroids:  # fallback
        embeddings = [units_to_embed[u] for u in units if u in units_to_embed]
        if embeddings:
            centroids.append(np.mean(embeddings, axis=0))
    return centroids

df["canonical_units"] = df["normalized_units"].apply(units_to_centroids_with_fallback)


# Cross reference with TMI

In [ ]:
thompson_df = pd.read_csv("aft-thompson.csv")
thompson_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1518 entries, 0 to 1517
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        1518 non-null   int64 
 1   atu_id            1518 non-null   object
 2   tale_title        1518 non-null   object
 3   provenance        1502 non-null   object
 4   notes             831 non-null    object
 5   source            1336 non-null   object
 6   text              1518 non-null   object
 7   data_source       1518 non-null   object
 8   date_obtained     1518 non-null   object
 9   extracted_motifs  1518 non-null   object
 10  motif_count       1518 non-null   int64 
dtypes: int64(2), object(9)
memory usage: 130.6+ KB


per atu

In [ ]:
import pandas as pd
import torch
from sentence_transformers import util

def compare_with_thompson(df, atu_to_tmi, model, reducer, level="extracted_motifs", sim_thr=0.4):

    reference_per_atu = (
        atu_to_tmi
        .groupby('atu_id')
        .agg({
            level: list,
            'extracted_motifs': list
        })
        .rename(columns={
            level: 'ref_texts',
            'thompson_motif_id': 'ref_ids'
        })
    )

    all_ref_texts = []
    atu_offsets = {}
    offset = 0

    for atu, row in reference_per_atu.iterrows():
        texts = row['ref_texts']
        if texts:
            all_ref_texts.extend(texts)
            atu_offsets[atu] = (offset, offset + len(texts))
            offset += len(texts)

    if not all_ref_texts:
        return pd.DataFrame()

    ref_embeddings_full = model.encode(
        all_ref_texts,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=128,
        device = "cuda"
    )
    if reducer is not None:
        ref_embeddings_reduced = reducer.transform(ref_embeddings_full)
    else:
        ref_embeddings_reduced = ref_embeddings_full

    ref_embeddings = {}
    for atu, (start, end) in atu_offsets.items():
        ref_embeddings[atu] = ref_embeddings_reduced[start:end]

    def flatten_vectors(list_of_lists):
        return [vec for sublist in list_of_lists if isinstance(sublist, list) for vec in sublist]

    df_grouped = (
        df.groupby("atu_id")["canonical_units"]
        .agg(flatten_vectors)
        .reset_index()
        .rename(columns={"canonical_units": "aggregated_canonical"})
    )

    comparison_df = df_grouped.merge(
        reference_per_atu.reset_index(),
        on='atu_id',
        how='inner'
    )

    results = []

    for _, row in comparison_df.iterrows():
        atu = row['atu_id']
        canon_vecs = row['aggregated_canonical']

        if canon_vecs is None or len(canon_vecs) == 0:
            results.append({
                'atu_id': atu,
                'n_stories': 0,
                'n_your_motifs': 0,
                'n_ref_motifs': len(row['ref_texts']),
                'recall': 0.0,
                'precision': 0.0
            })
            continue

        emb_yours = torch.tensor(canon_vecs, dtype=torch.float32)
        emb_ref = torch.tensor(ref_embeddings.get(atu, []), dtype=torch.float32)

        if emb_ref.numel() == 0:
            continue

        emb_yours = torch.nn.functional.normalize(emb_yours, p=2, dim=1)
        emb_ref = torch.nn.functional.normalize(emb_ref, p=2, dim=1)

        sim_matrix = util.cos_sim(emb_yours, emb_ref)

        max_per_ref = sim_matrix.max(dim=0).values
        recall = (max_per_ref >= sim_thr).float().mean().item()

        max_per_yours = sim_matrix.max(dim=1).values
        precision = (max_per_yours >= sim_thr).float().mean().item()

        results.append({
            'atu_id': atu,
            'n_stories': len(df[df['atu_id'] == atu]),
            'n_your_motifs': len(canon_vecs),
            'n_ref_motifs': len(row['ref_texts']),
            'recall': round(recall, 3),
            'precision': round(precision, 3)
        })

    results_df = pd.DataFrame(results)

    return results_df


In [ ]:
results_df = compare_with_thompson(
    df,
    thompson_df,
    model,
    reducer=svd,
    sim_thr=0.4
)

print(results_df.head())
print(results_df[['recall','precision']].mean())


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

/tmp/ipython-input-2777231228.py:83: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  emb_yours = torch.tensor(canon_vecs, dtype=torch.float32)


  atu_id  n_stories  n_your_motifs  n_ref_motifs  recall  precision
0      1          8             63             8   1.000      0.635
1    101          5             45             5   0.600      0.578
2    103          2             24             2   1.000      0.583
3   1030         11             76            11   0.909      0.763
4    105         13             89            13   0.615      0.730
recall       0.789167
precision    0.509100
dtype: float64


per story

In [ ]:
import pandas as pd
import torch
import ast
from sentence_transformers import util

def compare_with_thompson_by_title(df, thompson_df, model, reducer=None, sim_thr=0.4):
    match_col = "tale_title"

    def parse_gold_motifs(val):
        if pd.isna(val):
            return []
        if isinstance(val, str):
            val = val.strip()
            if val == "No motifs found." or val == "":
                return []
            if val.startswith('[') and val.endswith(']'):
                try:
                    return ast.literal_eval(val)
                except:
                    pass
            return [m.strip() for m in val.split('\n') if m.strip()]
        if isinstance(val, list):
            if len(val) == 1 and val[0] == "No motifs found.":
                return []
            return val
        return []

    thompson_df['gold_motifs_list'] = thompson_df['extracted_motifs'].apply(parse_gold_motifs)

    def flatten_vectors(list_of_lists):
        if not isinstance(list_of_lists, (list, pd.Series)):
            return []
        return [vec for sublist in list_of_lists if isinstance(sublist, list) for vec in sublist]

    df_grouped = (
        df.groupby(match_col)["canonical_units"]
        .agg(flatten_vectors)
        .reset_index()
        .rename(columns={"canonical_units": "aggregated_canonical"})
    )

    comparison_df = pd.merge(
        df_grouped,
        thompson_df[[match_col, 'gold_motifs_list']],
        on=match_col,
        how='inner'
    )

    if comparison_df.empty:
        return pd.DataFrame()

    all_gold_texts = []
    title_offsets = {}
    offset = 0

    for idx, row in comparison_df.iterrows():
        texts = row['gold_motifs_list']
        if texts:
            all_gold_texts.extend(texts)
            title_offsets[row[match_col]] = (offset, offset + len(texts))
            offset += len(texts)

    if all_gold_texts:
        ref_embeddings_full = model.encode(
            all_gold_texts,
            normalize_embeddings=True,
            show_progress_bar=True,
            batch_size=128,
            device="cuda"
        )
        if reducer is not None:
            ref_embeddings_reduced = reducer.transform(ref_embeddings_full)
        else:
            ref_embeddings_reduced = ref_embeddings_full
    else:
        ref_embeddings_reduced = []

    ref_embeddings = {}
    for title, (start, end) in title_offsets.items():
        ref_embeddings[title] = ref_embeddings_reduced[start:end]

    results = []

    for _, row in comparison_df.iterrows():
        title = row[match_col]
        canon_vecs = row['aggregated_canonical']
        gold_texts = row['gold_motifs_list']

        has_yours = canon_vecs is not None and len(canon_vecs) > 0
        has_gold = len(gold_texts) > 0

        if not has_yours and not has_gold:
            results.append({match_col: title, 'n_your_motifs': 0, 'n_ref_motifs': 0, 'recall': 1.0, 'precision': 1.0})
            continue
        elif not has_yours and has_gold:
            results.append({match_col: title, 'n_your_motifs': 0, 'n_ref_motifs': len(gold_texts), 'recall': 0.0, 'precision': 0.0})
            continue
        elif has_yours and not has_gold:
            results.append({match_col: title, 'n_your_motifs': len(canon_vecs), 'n_ref_motifs': 0, 'recall': 1.0, 'precision': 0.0})
            continue

        emb_yours = torch.tensor(canon_vecs, dtype=torch.float32)
        emb_ref = torch.tensor(ref_embeddings[title], dtype=torch.float32)

        emb_yours = torch.nn.functional.normalize(emb_yours, p=2, dim=1)
        emb_ref = torch.nn.functional.normalize(emb_ref, p=2, dim=1)

        sim_matrix = util.cos_sim(emb_yours, emb_ref)

        max_per_ref = sim_matrix.max(dim=0).values
        recall = (max_per_ref >= sim_thr).float().mean().item()

        max_per_yours = sim_matrix.max(dim=1).values
        precision = (max_per_yours >= sim_thr).float().mean().item()

        results.append({
            match_col: title,
            'n_your_motifs': len(canon_vecs),
            'n_ref_motifs': len(gold_texts),
            'recall': round(recall, 3),
            'precision': round(precision, 3)
        })

    return pd.DataFrame(results)

In [ ]:
results_df = compare_with_thompson_by_title(
    df,
    thompson_df,
    model,
    reducer=svd,
    sim_thr=0.4
)

print(results_df.head())
print(results_df[['recall','precision']].mean())

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

                                          tale_title  n_your_motifs  \
0                        A Blessed and Happie People              4   
1  A Brahmin Asks Two Parrots to Keep an Eye on H...              8   
2                 A Bridegroom for Miss Mole (Korea)              6   
3  A Child's Hand That Wrongly Attacked a Mother ...              9   
4          A Child's Thankfulness and Unthankfulness              2   

   n_ref_motifs  recall  precision  
0             1     0.0      0.000  
1             1     1.0      0.125  
2             1     1.0      0.167  
3             2     1.0      0.556  
4             1     0.0      0.000  
recall       0.699628
precision    0.334535
dtype: float64
